# Log4j AST Extraction (Notebook Wrapper)

This notebook runs and inspects the AST extraction pipeline implemented in `scripts/extract_log4j_ast.py`. The script remains the single source of truth, so notebook experiments and command-line runs produce the same artifacts.

The pipeline creates one filtered AST graph for each mapped Log4j Java class and exports GNN-ready tensors.

## 0) Setup Paths and Imports

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

# Resolve repo root whether the notebook is run from repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_log4j_ast.py'
AST_OUTPUT_DIR = REPO_ROOT / 'outputs' / 'log4j' / 'ast'

print('Repository root:', REPO_ROOT)
print('Extraction script:', SCRIPT_PATH)
print('AST output directory:', AST_OUTPUT_DIR)

Repository root: /Users/iman/Desktop/python_sdp_gnn
Extraction script: /Users/iman/Desktop/python_sdp_gnn/scripts/extract_log4j_ast.py
AST output directory: /Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ast


## 1) Run AST Extraction

The extractor reads `outputs/log4j/log4j_name_to_source_mapping.csv`, parses each mapped Java file, filters the AST to important syntax nodes, and reconnects retained nodes to their nearest retained ancestor.

It exports:
- readable graph JSON files
- structural node feature tensors
- exact AST node type id tensors
- edge index tensors
- a stable node type vocabulary
- a summary and markdown report

In [2]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)

AST extraction finished. graphs_generated=119 failures=0
index=/Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ast/graph_index.csv
summary=/Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ast/ast_summary.json
report=/Users/iman/Desktop/python_sdp_gnn/outputs/log4j/ast/ast_report.md



## 2) Inspect Extraction Summary

In [3]:
summary = json.loads((AST_OUTPUT_DIR / 'ast_summary.json').read_text())
summary

{'requested_mapped_samples': 119,
 'graphs_generated': 119,
 'parse_failures': 0,
 'total_nodes': 21465,
 'total_edges': 21346,
 'avg_nodes_per_graph': 180.3781512605042,
 'avg_edges_per_graph': 179.3781512605042,
 'structural_feature_dim': 3,
 'node_type_embedding_dim': 32,
 'model_node_feature_dim': 35,
 'node_type_vocab_size': 41,
 'fallback_graphs': 4,
 'important_node_types': ['Annotation',
  'Assignment',
  'BasicType',
  'BinaryOperation',
  'BlockStatement',
  'BreakStatement',
  'Cast',
  'CatchClause',
  'ClassCreator',
  'ClassDeclaration',
  'CompilationUnit',
  'ConstructorDeclaration',
  'ContinueStatement',
  'DoStatement',
  'EnumDeclaration',
  'FieldDeclaration',
  'ForStatement',
  'FormalParameter',
  'IfStatement',
  'Import',
  'InterfaceDeclaration',
  'Literal',
  'LocalVariableDeclaration',
  'MemberReference',
  'MethodDeclaration',
  'MethodInvocation',
  'PackageDeclaration',
  'ReferenceType',
  'ReturnStatement',
  'StatementExpression',
  'SuperMethodInvo

In [4]:
graph_index = pd.read_csv(AST_OUTPUT_DIR / 'graph_index.csv')
print('Generated graphs:', len(graph_index))
print()
print('Parser modes:')
print(graph_index['parser_mode'].value_counts())
graph_index.head()

Generated graphs: 119

Parser modes:
parser_mode
javalang    115
fallback      4
Name: count, dtype: int64


,name,source_path,num_nodes,num_edges,feature_dim,parser_mode,graph_json,x_npy,node_type_id_npy,edge_index_npy
0,org.apache.log4j.Appender,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,27,26,3,javalang,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
1,org.apache.log4j.AppenderSkeleton,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,220,219,3,javalang,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
2,org.apache.log4j.AsyncAppender,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,242,241,3,javalang,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
3,org.apache.log4j.BasicConfigurator,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,197,196,3,javalang,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
4,org.apache.log4j.Category,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,381,380,3,fallback,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...


## 3) Understand the Node Tensors

Each AST node is stored in two parts:

```text
node_type_id.npy -> exact AST syntax type id
x.npy            -> [depth, out_degree, has_identifier]
```

The node type embedding is not stored in the dataset. During GNN training, a learnable embedding layer converts each type id into 32 learned values. The model concatenates those values with the three extracted structural features:

```text
complete_node_x = [node_type_embedding[32], depth, out_degree, has_identifier]
```

The complete model-time feature vector therefore has `35` dimensions.

In [5]:
node_type_vocab = json.loads((AST_OUTPUT_DIR / 'node_type_vocab.json').read_text())
id_to_node_type = {node_type_id: node_type for node_type, node_type_id in node_type_vocab.items()}

print('Exact AST node type vocabulary size:', len(node_type_vocab))
pd.DataFrame(
    sorted(node_type_vocab.items(), key=lambda item: item[1]),
    columns=['node_type', 'node_type_id'],
)

Exact AST node type vocabulary size: 41


,node_type,node_type_id
0,Annotation,0
1,Assignment,1
2,BasicType,2
3,BinaryOperation,3
4,BlockStatement,4
5,BreakStatement,5
6,Cast,6
7,CatchClause,7
8,ClassCreator,8
9,ClassDeclaration,9


## 4) Inspect One Example Graph

In [6]:
EXAMPLE_CLASS = 'org.apache.log4j.Appender'
tensor_dir = AST_OUTPUT_DIR / 'tensors'

structural_x = np.load(tensor_dir / f'{EXAMPLE_CLASS}_x.npy')
node_type_ids = np.load(tensor_dir / f'{EXAMPLE_CLASS}_node_type_id.npy')
edge_index = np.load(tensor_dir / f'{EXAMPLE_CLASS}_edge_index.npy')

print('Class:', EXAMPLE_CLASS)
print('structural_x shape:', structural_x.shape)
print('node_type_ids shape:', node_type_ids.shape)
print('edge_index shape:', edge_index.shape)

Class: org.apache.log4j.Appender
structural_x shape: (27, 3)
node_type_ids shape: (27,)
edge_index shape: (2, 26)


In [7]:
node_preview = pd.DataFrame({
    'node_id': np.arange(len(node_type_ids)),
    'node_type_id': node_type_ids,
    'node_type': [id_to_node_type[int(node_type_id)] for node_type_id in node_type_ids],
    'depth': structural_x[:, 0],
    'out_degree': structural_x[:, 1],
    'has_identifier': structural_x[:, 2],
})
node_preview.head(15)

,node_id,node_type_id,node_type,depth,out_degree,has_identifier
0,0,10,CompilationUnit,0.0,5.0,0.0
1,1,26,PackageDeclaration,1.0,0.0,1.0
2,2,19,Import,1.0,0.0,1.0
3,3,19,Import,1.0,0.0,1.0
4,4,19,Import,1.0,0.0,1.0
5,5,20,InterfaceDeclaration,1.0,9.0,1.0
6,6,24,MethodDeclaration,2.0,1.0,1.0
7,7,17,FormalParameter,3.0,1.0,1.0
8,8,27,ReferenceType,4.0,0.0,1.0
9,9,24,MethodDeclaration,2.0,0.0,1.0


In [8]:
edge_preview = pd.DataFrame({
    'source_node_id': edge_index[0],
    'target_node_id': edge_index[1],
})
edge_preview.head(20)

,source_node_id,target_node_id
0,0,1
1,0,2
2,0,3
3,0,4
4,0,5
5,5,6
6,6,7
7,7,8
8,5,9
9,5,10


## 5) Assemble Features Inside the GNN

The extraction output is designed for a trainable node encoder. The embedding values belong to the model because they must be updated by gradient descent while learning defect prediction.

A minimal PyTorch-style encoder is:

```python
import torch
from torch import nn

class AstNodeEncoder(nn.Module):
    def __init__(self, node_type_vocab_size: int, embedding_dim: int = 32):
        super().__init__()
        self.node_type_embedding = nn.Embedding(node_type_vocab_size, embedding_dim)

    def forward(self, node_type_id: torch.Tensor, structural_x: torch.Tensor):
        type_x = self.node_type_embedding(node_type_id)
        return torch.cat([type_x, structural_x], dim=1)

# complete_x shape: [num_nodes, 35]
```

Pass `complete_x` and `edge_index` to the first GNN layer. After message passing, pool node embeddings into one graph embedding for the Java class and pass that graph embedding to a defect classifier.

## 6) Inspect the Generated Report

In [9]:
report_text = (AST_OUTPUT_DIR / 'ast_report.md').read_text()

try:
    from IPython.display import Markdown, display
    display(Markdown(report_text))
except ImportError:
    print(report_text)

# AST Extraction Report for Log4j 1.0

## 1. Objective
This report describes the AST graph extraction step for the Log4j 1.0 PROMISE dataset used in the multi-view software defect prediction pipeline. The goal is to create one AST graph for each dataset row that has a matched Java source file, then store the result in a format that can be used directly by a Graph Neural Network.

## 2. Input Data
- Dataset: `projects/log4j/log4j-1.0.csv`
- Preprocessed dataset: `outputs/log4j/log4j_preprocessed_standard.csv`
- Class-to-source mapping: `outputs/log4j/log4j_name_to_source_mapping.csv`
- Mapped source files used for AST extraction: 119

## 3. Extraction Pipeline
- Each mapped Java file is parsed with `javalang`.
- Only important AST node types are kept to reduce noise and graph size.
- Removed intermediate nodes are bypassed by reconnecting each kept node to the nearest kept ancestor.
- Parent-child AST relations are saved as directed edges from parent node to child node.
- Four legacy Java files required the deterministic fallback parser and are marked with `parser_mode=fallback`.

## 4. Selected AST Node Types
The extractor keeps nodes that are useful for defect prediction: declarations, control-flow statements, expressions, method calls, variable references, literals, and type information. This keeps the graph focused on program structure and behavior instead of low-value parser details.

| Group | Examples | Why it matters |
| --- | --- | --- |
| Declaration | `ClassDeclaration`, `MethodDeclaration`, `FieldDeclaration` | Captures class and API structure. |
| Control | `IfStatement`, `ForStatement`, `TryStatement`, `ReturnStatement` | Captures branching, loops, exceptions, and exits. |
| Expression | `MethodInvocation`, `Assignment`, `BinaryOperation`, `MemberReference` | Captures behavior inside methods. |
| Type | `ReferenceType`, `BasicType`, `Annotation` | Preserves type-level context. |
| Literal | `Literal` | Preserves constants and string/numeric usage. |

## 5. What the Tensor Files Mean
A tensor is a numeric array used by machine learning frameworks. In this project, each AST graph is stored in tensor files so a GNN can process it efficiently.

| File | Shape | Meaning |
| --- | --- | --- |
| `*_x.npy` | `[num_nodes, 3]` | Structural syntax features: depth, out-degree, and has-identifier. |
| `*_node_type_id.npy` | `[num_nodes]` | Exact AST node type ids consumed by a trainable embedding layer. |
| `*_edge_index.npy` | `[2, num_edges]` | Graph connectivity. Row 0 stores source node ids; row 1 stores target node ids. |

The extractor stores syntax-focused inputs. During model training, transform each `node_type_id` with `Embedding(vocab_size, 32)` and concatenate the result with the three structural features. The GNN then receives 35 values per node.

| Model input index range | Feature | Description |
| --- | --- | --- |
| `0-31` | Learned node type embedding | Trainable representation of the exact retained AST node type. |
| `32` | Depth | How deep the node is in the AST. |
| `33` | Out-degree | How many kept child nodes it has. |
| `34` | Has identifier | Whether the node carries identifier-like text. |

Exact node types are not collapsed into broad groups. Each type has a stable id in `node_type_vocab.json`, for example `MethodDeclaration`, `IfStatement`, `Assignment`, and `MethodInvocation`. The embedding table belongs in the training model and is learned from the defect prediction objective.

Example for one graph:

```python
import numpy as np

x = np.load('outputs/log4j/ast/tensors/org.apache.log4j.AppenderSkeleton_x.npy')
node_type_id = np.load('outputs/log4j/ast/tensors/org.apache.log4j.AppenderSkeleton_node_type_id.npy')
edge_index = np.load('outputs/log4j/ast/tensors/org.apache.log4j.AppenderSkeleton_edge_index.npy')

print(x.shape)            # (num_nodes, 3)
print(node_type_id.shape) # (num_nodes,)
print(edge_index.shape) # (2, num_edges)
```

## 6. How to Use the AST Tensors in a GNN
The complete node representation is assembled inside the model, not during AST extraction. This is necessary because the 32 embedding values must be learned from the defect prediction task. They are model parameters, while depth, out-degree, and has-identifier are fixed facts extracted from the source code.

For a node with `node_type_id = 24` and structural features `[2, 1, 1]`:

```text
node_type_id = 24
structural_x = [depth=2, out_degree=1, has_identifier=1]
type_embedding = embedding_layer(24)  # learned vector with 32 values
complete_node_x = concat(type_embedding, structural_x)  # 35 values
```

At model level, the processing steps are:

1. Load `node_type_id`, structural `x`, and `edge_index` for each AST graph.
2. Convert every `node_type_id` to a learned 32-dimensional embedding.
3. Concatenate each embedding with its three structural values.
4. Pass the resulting 35-dimensional node vectors and AST edges into GNN layers.
5. Pool all node representations into one graph representation for the Java class.
6. Feed the graph representation into a classifier to predict whether the class is defective.

Minimal PyTorch-style model preparation:

```python
import torch
from torch import nn

class AstNodeEncoder(nn.Module):
    def __init__(self, node_type_vocab_size: int, embedding_dim: int = 32):
        super().__init__()
        self.node_type_embedding = nn.Embedding(node_type_vocab_size, embedding_dim)

    def forward(self, node_type_id: torch.Tensor, structural_x: torch.Tensor):
        type_x = self.node_type_embedding(node_type_id)
        return torch.cat([type_x, structural_x], dim=1)

# node_type_id: [num_nodes]
# structural_x: [num_nodes, 3]
# complete_x:   [num_nodes, 35]
complete_x = AstNodeEncoder(node_type_vocab_size=41)(node_type_id, structural_x)
```

The returned `complete_x` is the node feature matrix passed to the first GNN layer together with `edge_index`. The future CFG and NDG views should follow the same principle: keep node attributes compact and represent program behavior primarily through graph edges.

## 7. Output Files
- `outputs/log4j/ast/graph_index.csv`: index of all generated graphs and tensor paths.
- `outputs/log4j/ast/graphs/*.json`: readable graph files with node metadata and edge lists.
- `outputs/log4j/ast/tensors/*_x.npy`: structural syntax feature tensors.
- `outputs/log4j/ast/tensors/*_node_type_id.npy`: exact node type id tensors for trainable embeddings.
- `outputs/log4j/ast/tensors/*_edge_index.npy`: edge index tensors.
- `outputs/log4j/ast/node_type_vocab.json`: stable mapping from exact AST node type to id.
- `outputs/log4j/ast/ast_summary.json`: global extraction statistics.
- `outputs/log4j/ast/parse_failures.json`: parse failure log; currently empty.

## 8. Results

| Metric | Value |
| --- | ---: |
| Mapped samples requested | 119 |
| Graphs generated | 119 |
| Parse failures | 0 |
| Fallback graphs | 4 |
| Total nodes | 21465 |
| Total edges | 21346 |
| Average nodes per graph | 180.38 |
| Average edges per graph | 179.38 |
| Extracted structural feature dimension | 3 |
| Learned node type embedding dimension | 32 |
| Model node feature dimension after concatenation | 35 |
| Exact AST node type vocabulary size | 41 |

## 9. Top Node Types
- MemberReference: 3451
- MethodInvocation: 2417
- Literal: 2144
- ReferenceType: 1950
- StatementExpression: 1906
- BinaryOperation: 1133
- VariableDeclarator: 1008
- Assignment: 788
- MethodDeclaration: 688
- Import: 657

## 10. Notes and Limitations
The AST view captures syntactic structure but does not directly encode runtime execution order or data dependencies. CFG and NDG extraction should be added as separate graph views and joined later by the same class name or graph index row.
The node feature design intentionally stays syntax-focused. Future graph views should represent program behavior through typed edges, such as control-flow and dependency relations, instead of adding behavior-specific values to AST nodes.

The fallback graphs are useful for keeping the dataset complete, but they are less precise than the `javalang` graphs. For experiments, keep `parser_mode` as metadata so these four samples can be included, excluded, or analyzed separately.

## 11. Sample Visualizations
The following Mermaid diagrams show compact views of the first 45 kept AST nodes for three example classes.
### 1. org.apache.log4j.helpers.ISO8601DateFormat
```mermaid
graph TD
  N0["CompilationUnit#0"]
  N1["PackageDeclaration#1"]
  N2["Import#2"]
  N3["Import#3"]
  N4["Import#4"]
  N5["Import#5"]
  N6["Import#6"]
  N7["Import#7"]
  N8["Import#8"]
  N9["ClassDeclaration#9"]
  N10["ConstructorDeclaration#10"]
  N11["ConstructorDeclaration#11"]
  N12["FormalParameter#12"]
  N13["ReferenceType#13"]
  N14["StatementExpression#14"]
  N15["MemberReference#15"]
  N16["MethodDeclaration#16"]
  N17["ReferenceType#17"]
  N18["FormalParameter#18"]
  N19["ReferenceType#19"]
  N20["FormalParameter#20"]
  N21["ReferenceType#21"]
  N22["FormalParameter#22"]
  N23["ReferenceType#23"]
  N24["StatementExpression#24"]
  N25["MethodInvocation#25"]
  N26["MemberReference#26"]
  N27["LocalVariableDeclaration#27"]
  N28["BasicType#28"]
  N29["VariableDeclarator#29"]
  N30["MethodInvocation#30"]
  N31["MemberReference#31"]
  N32["StatementExpression#32"]
  N33["MethodInvocation#33"]
  N34["MemberReference#34"]
  N35["LocalVariableDeclaration#35"]
  N36["ReferenceType#36"]
  N37["VariableDeclarator#37"]
  N38["SwitchStatement#38"]
  N39["MethodInvocation#39"]
  N40["MemberReference#40"]
  N41["SwitchStatementCase#41"]
  N42["MemberReference#42"]
  N43["StatementExpression#43"]
  N44["Assignment#44"]
  N0 --> N1
  N0 --> N2
  N0 --> N3
  N0 --> N4
  N0 --> N5
  N0 --> N6
  N0 --> N7
  N0 --> N8
  N0 --> N9
  N9 --> N10
  N9 --> N11
  N11 --> N12
  N12 --> N13
  N11 --> N14
  N14 --> N15
  N9 --> N16
  N16 --> N17
  N16 --> N18
  N18 --> N19
  N16 --> N20
  N20 --> N21
  N16 --> N22
  N22 --> N23
  N16 --> N24
  N24 --> N25
  N25 --> N26
  N16 --> N27
  N27 --> N28
  N27 --> N29
  N29 --> N30
  N30 --> N31
  N16 --> N32
  N32 --> N33
  N33 --> N34
  N16 --> N35
  N35 --> N36
  N35 --> N37
  N16 --> N38
  N38 --> N39
  N39 --> N40
  N38 --> N41
  N41 --> N42
  N41 --> N43
  N43 --> N44
```
### 2. org.apache.log4j.xml.Transform
```mermaid
graph TD
  N0["CompilationUnit#0"]
  N1["PackageDeclaration#1"]
  N2["Import#2"]
  N3["Import#3"]
  N4["Import#4"]
  N5["Import#5"]
  N6["Import#6"]
  N7["Import#7"]
  N8["Import#8"]
  N9["Import#9"]
  N10["Import#10"]
  N11["Import#11"]
  N12["Import#12"]
  N13["Import#13"]
  N14["Import#14"]
  N15["Import#15"]
  N16["Import#16"]
  N17["Import#17"]
  N18["Import#18"]
  N19["Import#19"]
  N20["Import#20"]
  N21["Import#21"]
  N22["Import#22"]
  N23["Import#23"]
  N24["Import#24"]
  N25["Import#25"]
  N26["Import#26"]
  N27["Import#27"]
  N28["Import#28"]
  N29["Import#29"]
  N30["ClassDeclaration#30"]
  N31["MethodDeclaration#31"]
  N32["FormalParameter#32"]
  N33["ReferenceType#33"]
  N34["StatementExpression#34"]
  N35["MethodInvocation#35"]
  N36["StatementExpression#36"]
  N37["MethodInvocation#37"]
  N38["Literal#38"]
  N39["LocalVariableDeclaration#39"]
  N40["ReferenceType#40"]
  N41["VariableDeclarator#41"]
  N42["MethodInvocation#42"]
  N43["Literal#43"]
  N44["LocalVariableDeclaration#44"]
  N0 --> N1
  N0 --> N2
  N0 --> N3
  N0 --> N4
  N0 --> N5
  N0 --> N6
  N0 --> N7
  N0 --> N8
  N0 --> N9
  N0 --> N10
  N0 --> N11
  N0 --> N12
  N0 --> N13
  N0 --> N14
  N0 --> N15
  N0 --> N16
  N0 --> N17
  N0 --> N18
  N0 --> N19
  N0 --> N20
  N0 --> N21
  N0 --> N22
  N0 --> N23
  N0 --> N24
  N0 --> N25
  N0 --> N26
  N0 --> N27
  N0 --> N28
  N0 --> N29
  N0 --> N30
  N30 --> N31
  N31 --> N32
  N32 --> N33
  N31 --> N34
  N34 --> N35
  N31 --> N36
  N36 --> N37
  N37 --> N38
  N31 --> N39
  N39 --> N40
  N39 --> N41
  N41 --> N42
  N42 --> N43
  N31 --> N44
```
### 3. org.apache.log4j.helpers.AppenderAttachableImpl
```mermaid
graph TD
  N0["CompilationUnit#0"]
  N1["PackageDeclaration#1"]
  N2["Import#2"]
  N3["Import#3"]
  N4["Import#4"]
  N5["Import#5"]
  N6["Import#6"]
  N7["ClassDeclaration#7"]
  N8["FieldDeclaration#8"]
  N9["ReferenceType#9"]
  N10["VariableDeclarator#10"]
  N11["MethodDeclaration#11"]
  N12["FormalParameter#12"]
  N13["ReferenceType#13"]
  N14["IfStatement#14"]
  N15["BinaryOperation#15"]
  N16["MemberReference#16"]
  N17["Literal#17"]
  N18["ReturnStatement#18"]
  N19["IfStatement#19"]
  N20["BinaryOperation#20"]
  N21["MemberReference#21"]
  N22["Literal#22"]
  N23["BlockStatement#23"]
  N24["StatementExpression#24"]
  N25["Assignment#25"]
  N26["MemberReference#26"]
  N27["ClassCreator#27"]
  N28["ReferenceType#28"]
  N29["Literal#29"]
  N30["IfStatement#30"]
  N31["MethodInvocation#31"]
  N32["MemberReference#32"]
  N33["StatementExpression#33"]
  N34["MethodInvocation#34"]
  N35["MemberReference#35"]
  N36["MethodDeclaration#36"]
  N37["BasicType#37"]
  N38["FormalParameter#38"]
  N39["ReferenceType#39"]
  N40["LocalVariableDeclaration#40"]
  N41["BasicType#41"]
  N42["VariableDeclarator#42"]
  N43["Literal#43"]
  N44["LocalVariableDeclaration#44"]
  N0 --> N1
  N0 --> N2
  N0 --> N3
  N0 --> N4
  N0 --> N5
  N0 --> N6
  N0 --> N7
  N7 --> N8
  N8 --> N9
  N8 --> N10
  N7 --> N11
  N11 --> N12
  N12 --> N13
  N11 --> N14
  N14 --> N15
  N15 --> N16
  N15 --> N17
  N14 --> N18
  N11 --> N19
  N19 --> N20
  N20 --> N21
  N20 --> N22
  N19 --> N23
  N23 --> N24
  N24 --> N25
  N25 --> N26
  N25 --> N27
  N27 --> N28
  N27 --> N29
  N11 --> N30
  N30 --> N31
  N31 --> N32
  N30 --> N33
  N33 --> N34
  N34 --> N35
  N7 --> N36
  N36 --> N37
  N36 --> N38
  N38 --> N39
  N36 --> N40
  N40 --> N41
  N40 --> N42
  N42 --> N43
  N36 --> N44
```
